# Lecture 8: Linear Models, LDA, and Logistic Regression

This Notebook accompanies the supplied experiment results; its code cells have no saved execution record. Stages cover A: data audit, B: development comparisons, C: thresholds and freezing, and D: locked-test evaluation. `code/run_analysis.py` implements the full experiment and 27 plots. Read `data/heart.csv` from this week's directory; exclude `num`, `site`, and record identifiers from model inputs. **Run D only after confirming the frozen protocol.** The supplied decision is recorded in `results/frozen_protocol_before_test.json`.

## A. Data Audit: Types, Missingness, Labels, and Sites

In [ ]:
from pathlib import Path
import sys
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'assignments/week-08-topic']
ROOT = next((p for p in candidates if (p / 'code/run_analysis.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this Notebook from the repository root, week-08 directory, or its notebooks directory.')
sys.path.insert(0, str(ROOT / 'code'))
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from run_analysis import FEATURES, NUM, CAT
raw=pd.read_csv(ROOT / 'data/heart.csv')
assert raw.shape==(270,18)
print('Target counts (1=positive, 0=negative):',raw.heart_disease_present.value_counts().to_dict())
print('Original missing values:\n',raw[FEATURES].isna().sum().sort_values(ascending=False))
print('Cholesterol=0:',(raw.chol==0).sum())
print('Site composition:\n',pd.crosstab(raw.site,raw.heart_disease_present))
X=raw[FEATURES].copy()
X.loc[X.chol==0,'chol']=np.nan
y=raw.heart_disease_present.astype(int)
dev_idx,test_idx=train_test_split(np.arange(len(y)),test_size=.2,stratify=y,random_state=2026)
Xdev=X.iloc[dev_idx];ydev=y.iloc[dev_idx];Xtest=X.iloc[test_idx]
print('Development records:',len(dev_idx),'Locked-test records:',len(test_idx))

**Audit conclusion:** The course plan's claim of no missing values does not match the supplied `heart.csv`. There are 557 missing input cells and 42 zero cholesterol values treated as potentially missing, giving 599 missing cells for analysis. Site may confound source differences and is used only for exploratory auditing.

## B. Fold-Local Training and Development OOF Predictions

In [ ]:
from run_analysis import processor,make_pipeline,algos,predict_metrics,assess_cost
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from sklearn.base import clone
pre=processor().fit(Xdev)
q=pre.transform(Xdev).shape[1]
print('Encoded dimension q=',q,'Nominal logistic parameters=',q+1,'LDA mean parameters=',2*q,'Shared covariance parameters=',q*(q+1)//2)
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=2026)
prob={}
for name in ['Prevalence baseline','Gaussian NB','LDA','Shrinkage LDA','Logistic','Random Forest','XGBoost','RBF-SVM']:
    pr=np.zeros(len(ydev))
    for tr,va in cv.split(Xdev,ydev):
        model=make_pipeline(name,clone(algos()[name]))
        model.fit(Xdev.iloc[tr],ydev.iloc[tr]);pr[va]=model.predict_proba(Xdev.iloc[va])[:,1]
    prob[name]=pr
    print(name,'OOF AP=',round(average_precision_score(ydev,pr),4))

## C. Cost-Based Thresholds and the Frozen Protocol

In [ ]:
from sklearn.metrics import brier_score_loss
thresholds={}; cost_rows=[]
for name,pr in prob.items():
    t,curve=assess_cost(ydev,pr)
    thresholds[name]=t
    cost_rows.append((name,t,curve.cost.min(),brier_score_loss(ydev,pr)))
print(pd.DataFrame(cost_rows,columns=['model','threshold','cost','Brier']).to_string(index=False))
# Select only between Logistic and Shrinkage LDA using development error cost.
cands=[r for r in cost_rows if r[0] in ['Logistic','Shrinkage LDA']]
cands.sort(key=lambda x:(x[2],x[3]))
chosen= cands[0][0];chosen_t=cands[0][1]
print('Frozen model:',chosen,'Threshold:',chosen_t)
assert chosen=='Shrinkage LDA' and chosen_t==0.15


**Decision rule:** Prespecify Logistic and Shrinkage LDA as the primary candidates. Select by minimum development OOF cost `5×FN+FP`, breaking ties by Brier score. More complex models are supplementary comparisons. Do not change models, thresholds, or feature processing based on final test scores.

## D. Locked Test and Complete Figures (Run Only After C)

This stage reruns the experiment and overwrites matching outputs in this week's `results/` and `figures/` directories.

In [ ]:
from pathlib import Path
from subprocess import run
import sys
# The full script repeats development selection before evaluating the locked test.
result=run([sys.executable,str(ROOT/'code/run_analysis.py'),'--data',str(ROOT/'data/heart.csv'),'--out',str(ROOT)],check=True)
summary=pd.read_csv(ROOT / 'results/locked_test_model_metrics.csv')
print(summary[['model','ROC_AUC','AP','Brier','Sensitivity','Specificity','FN','FP','Cost','Threshold']].to_string(index=False))
print('Saved PNG figures (expected 27):',len(list((ROOT / 'figures').glob('*.png'))))

## E. Main Conclusions and CERB

**Claim:** Under the assumed cost of missed disease, Shrinkage LDA is selected by the development OOF cost rule, although its difference from Logistic is small.

**Evidence:** Development costs are 82 and 83, respectively. Among 54 locked-test records, Shrinkage LDA has ROC-AUC approximately 0.818, AP approximately 0.773, sensitivity approximately 0.917, FN=2, and FP=16.

**Reasoning:** LDA obtains posteriors from class-conditional Gaussian distributions and priors; Logistic directly fits conditional probabilities. LDA estimates more covariance parameters, while mixed-type inputs and site-related missingness strain its assumptions.

**Boundary:** These are curated, small-sample teaching data. Internal results do not establish clinical validity or causal feature effects. Test results must not be used to select the algorithm again.

The supplied probability metrics come from saved experiment results. Figures are in `figures/`; algorithm and plotting settings are in `code/run_analysis.py`. Submission materials may include the worksheet Word document, two-column course report, and figure atlas PDF. Original reports and existing images retain their source language; rerunning the translated script produces English plot labels.